In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# ML Pipeline & Metrics
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

print("All dependencies loaded successfully.")

All dependencies loaded successfully.


In [2]:
# 1. Path to your baseline CSV dataset
CSV_PATH = r"C:\Users\Miss Jessy\Downloads\customer_support_500_cleaned.csv"

try:
    df_csv = pd.read_csv(CSV_PATH, encoding='utf-8')
    # Standardize column names
    rename_dict = {
        'queries': 'question', 
        'query': 'question',
        'answers': 'answer',
        'response': 'answer',
        'responses': 'answer', 
        'categories': 'category',
        }
    df_csv = df_csv.rename(columns=rename_dict)
    
    df_csv = df_csv[['question', 'answer', 'category']].dropna(subset=['question', 'answer'])
    print(f"Loaded {len(df_csv)} baseline rows from CSV.")
    display(df_csv.head())
    
except Exception as e:
    print(f"CSV load warning: {e}")
    df_csv = pd.DataFrame(columns=['question', 'answer', 'category'])

Loaded 500 baseline rows from CSV.


,question,answer,category
0,Sample customer question 1?,Sample answer 1.,Category2
1,Sample customer question 2?,Sample answer 2.,Category3
2,Sample customer question 3?,Sample answer 3.,Category4
3,Sample customer question 4?,Sample answer 4.,Category5
4,Sample customer question 5?,Sample answer 5.,Category1


In [4]:
# 2. Database connection settings
DB_USER = "postgres"
DB_PASSWORD = "2589Mteja"  
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "mteja_ai_db"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    
    query = "SELECT question, answer, category FROM training_data WHERE verified = TRUE;"
    df_db = pd.read_sql(query, engine)
    print(f"Loaded {len(df_db)} verified dynamic rows from PostgreSQL.")
except Exception as e:
    print(f"Database fetch warning: {e}")
    df_db = pd.DataFrame(columns=['question', 'answer', 'category'])

Loaded 0 verified dynamic rows from PostgreSQL.


In [5]:
# 1. make sure both DataFrames have the same columns for merging
df_csv = df_csv[['question', 'answer']].copy() if 'question' in df_csv.columns and 'answer' in df_csv.columns else pd.DataFrame(columns=['question', 'answer'])
df_db = df_db[['question', 'answer']].copy() if 'question' in df_db.columns and 'answer' in df_db.columns else pd.DataFrame(columns=['question', 'answer'])

# 2. combine the two DataFrames into one, ignoring the index to avoid duplicate indices
df_combined = pd.concat([df_csv, df_db], ignore_index=True)

# 3. log the number of missing values and total rows before cleaning
print("--- MISSING VALUES BEFORE CLEANING ---")
print(df_combined.isnull().sum())
print("\nTotal rows before cleaning:", len(df_combined))

# 4. clean the 'question' and 'answer' columns: lowercase, strip whitespace, and fill NaN with empty strings
df_combined['question_clean'] = df_combined['question'].fillna("").astype(str).str.lower().str.strip()
df_combined['answer_clean'] = df_combined['answer'].fillna("").astype(str).str.strip()

# 5. filter out rows where either the cleaned question or answer is empty
df_cleaned = df_combined[
    (df_combined['question_clean'] != "") & 
    (df_combined['answer_clean'] != "")
].copy()

# 6. drop duplicates based on the cleaned question, keeping the last occurrence
df_final = df_cleaned.drop_duplicates(subset=['question_clean'], keep='last').copy()

# 7. log the number of missing values and total rows after cleaning
print("\n--- CLEANING SUMMARY ---")
print("Total rows after cleaning & deduplication:", len(df_final))

# 8. display the first few rows of the cleaned DataFrame
X = df_final['question_clean']
y = df_final['answer_clean']

display(df_final[['question_clean', 'answer_clean']].head())

--- MISSING VALUES BEFORE CLEANING ---
question    0
answer      0
dtype: int64

Total rows before cleaning: 500

--- CLEANING SUMMARY ---
Total rows after cleaning & deduplication: 500


,question_clean,answer_clean
0,sample customer question 1?,Sample answer 1.
1,sample customer question 2?,Sample answer 2.
2,sample customer question 3?,Sample answer 3.
3,sample customer question 4?,Sample answer 4.
4,sample customer question 5?,Sample answer 5.


In [6]:
# Combine CSV data and DB data
df_combined = pd.concat([df_csv, df_db], ignore_index=True)

# Preprocess & remove exact duplicate questions
df_combined['question_clean'] = df_combined['question'].astype(str).str.lower().str.strip()
df_combined['answer_clean'] = df_combined['answer'].astype(str).str.strip()
if 'category' in df_combined.columns:
    df_combined['category_clean'] = df_combined['category'].astype(str).str.strip()

# Keep the latest verified database answers if duplicates exist
df_final = df_combined.drop_duplicates(subset=['question_clean'], keep='last').copy()

X = df_final['question_clean']
y = df_final['answer_clean']

print(f"Total training dataset size after merging and deduplication: {len(X)}")

Total training dataset size after merging and deduplication: 500


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Tengeneza na ufundishe Vectorizer kwenye maswali yote
vectorizer = TfidfVectorizer(ngram_range=(1, 3), analyzer='word')

# Transform maswali ya CSV + DB kuwa Matrix ya Namba (Vectors)
X_vectors = vectorizer.fit_transform(df_final['question_clean'])
y_answers = df_final['answer_clean'].values

print("Vectorization imekamilika kwa mafanikio!")

# 2. Function ya Kutabiri Jibu na Confidence Score
def predict_intent(user_query, threshold=0.15):
    clean_q = str(user_query).lower().strip()
    query_vector = vectorizer.transform([clean_q])
    
    # Piga hesabu ya Similarity kati ya swali jipya na maswali yote 500
    similarities = cosine_similarity(query_vector, X_vectors)[0]
    best_match_idx = np.argmax(similarities)
    best_score = similarities[best_match_idx]
    
    if best_score >= threshold:
        return {
            "answer": y_answers[best_match_idx],
            "matched_question": df_final['question_clean'].iloc[best_match_idx],
            "confidence": round(float(best_score) * 100, 2)
        }
    else:
        return {
            "answer": "Samahani, sijaelewa swali lako. Tafadhali jaribu kuuliza kwa njia nyingine.",
            "matched_question": None,
            "confidence": round(float(best_score) * 100, 2)
        }

# 3. Jaribu tena Maswali ya Nje (External Prediction Test)
test_queries = [
    "sample customer question 1?",
    "sample customer question 3",
    "Habari, naomba msaada wa akaunti"
]

print("\n--- EXTERNAL PREDICTION TEST (COSINE SIMILARITY) ---")
for query in test_queries:
    res = predict_intent(query)
    print(f"Query: '{query}'")
    print(f"Matched Q: '{res['matched_question']}'")
    print(f"Predicted Answer: '{res['answer']}'")
    print(f"Confidence: {res['confidence']}%\n" + "-"*50)

Vectorization imekamilika kwa mafanikio!

--- EXTERNAL PREDICTION TEST (COSINE SIMILARITY) ---
Query: 'sample customer question 1?'
Matched Q: 'sample customer question 1?'
Predicted Answer: 'Sample answer 1.'
Confidence: 100.0%
--------------------------------------------------
Query: 'sample customer question 3'
Matched Q: 'sample customer question 1?'
Predicted Answer: 'Sample answer 1.'
Confidence: 100.0%
--------------------------------------------------
Query: 'Habari, naomba msaada wa akaunti'
Matched Q: 'None'
Predicted Answer: 'Samahani, sijaelewa swali lako. Tafadhali jaribu kuuliza kwa njia nyingine.'
Confidence: 0.0%
--------------------------------------------------


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# 1. Piga hesabu ya Predictions kwa maswali YOTE 500 yaliyopo kwenye dataset
y_true = df_final['answer_clean'].values
y_pred = []

# Tumia Cosine Similarity kupata jibu la kila swali
cosine_sim_matrix = cosine_similarity(X_vectors, X_vectors)

for i in range(len(df_final)):
    # Pata index ya swali linalofanana zaidi (Top match)
    best_idx = np.argmax(cosine_sim_matrix[i])
    y_pred.append(y_answers[best_idx])

# 2. Piga hesabu za Metrics (Accuracy & Weighted F1 Score)
acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print("--- MODEL EVALUATION METRICS (VECTOR SEARCH) ---")
print(f"Overall Accuracy: {acc * 100:.2f}%")
print(f"Weighted Precision: {precision * 100:.2f}%")
print(f"Weighted Recall: {recall * 100:.2f}%")
print(f"Weighted F1 Score: {f1 * 100:.2f}%")

--- MODEL EVALUATION METRICS (VECTOR SEARCH) ---
Overall Accuracy: 98.40%
Weighted Precision: 98.22%
Weighted Recall: 98.40%
Weighted F1 Score: 98.24%


In [16]:
import os
import joblib

# Chukua vitu vyote vinavyohitajika kwa ajili ya FastAPI Inference
model_artifact = {
    "vectorizer": vectorizer,
    "X_vectors": X_vectors,
    "y_answers": y_answers,
    "df_data": df_final[['question_clean', 'answer_clean']]
}

output_dir = r"C:\Users\Miss Jessy\OneDrive\Desktop\Mteja-AI\backend\models"
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, "intent_classifier.pkl")
joblib.dump(model_artifact, model_path)

print(f"Adaptive Similarity Model imehifadhiwa kwa mafanikio hapa:\n{model_path}")

Adaptive Similarity Model imehifadhiwa kwa mafanikio hapa:
C:\Users\Miss Jessy\OneDrive\Desktop\Mteja-AI\backend\models\intent_classifier.pkl
